In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications import resnet50
from tensorflow.keras.preprocessing import image
from lime import lime_image
from skimage.segmentation import mark_boundaries

# 1. Load the pre-trained Keras model
model = resnet50.ResNet50(weights='imagenet')

# 2. Preprocess the image
def load_and_preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    # We don't preprocess yet because LIME needs the raw image 
    # to perturb it; we preprocess inside the prediction function.
    return x, img

# Replace 'path_to_image.jpg' with your local image file
img_array, img_raw = load_and_preprocess_image('dog_image.jpg')

# 3. Create a prediction wrapper
# LIME needs a function that takes a numpy array and returns probabilities
def batch_predict(images):
    # Apply ResNet50 specific preprocessing (scaling/mean subtraction)
    preprocessed_images = resnet50.preprocess_input(images.copy())
    return model.predict(preprocessed_images)

# 4. Initialize LIME Image Explainer
explainer = lime_image.LimeImageExplainer()

# 5. Generate Explanation
# hide_color=0 makes the 'off' super-pixels gray
explanation = explainer.explain_instance(img_array[0].astype('double'), 
                                         batch_predict, 
                                         top_labels=5, 
                                         hide_color=0, 
                                         num_samples=1000)

# 6. Visualize the positive segments (Pros)
temp, mask = explanation.get_image_and_mask(explanation.top_labels[0], 
                                            positive_only=True, 
                                            num_features=5, 
                                            hide_rest=True)

plt.imshow(mark_boundaries(temp / 255.0, mask))
plt.title("LIME Explanation for Top Class")
plt.show()